# Entrega 2 — Comparación de familias y validación
## Predicción del precio de bolsa · Mercado Eléctrico Colombiano

**Aprendizaje de Máquina Aplicado · Profesor Marco Terán · Universidad EAFIT · 2026**

---

> **Propósito de este notebook**
>
> Demostrar una comparación técnica seria entre familias de modelos aplicando un protocolo de validación honesto para series de tiempo. Cada sección incluye el fundamento teórico que justifica cada decisión, conectando las notas de clase con los datos reales del sistema eléctrico colombiano.
>
> **Regla central:** *train aprende, validation decide, test estima al final.*

## Tabla de contenidos
1. [Contrato metodológico](#contrato)
2. [Setup y datos](#setup)
3. [Feature engineering y rezagos temporales](#features)
4. [Protocolo de validación — TimeSeriesSplit](#cv)
5. [Baselines — referencia mínima exigible](#baselines)
6. [Familias candidatas](#familias)
7. [HPO — búsqueda de hiperparámetros dentro de validación](#hpo)
8. [Comparación final](#comparacion)
9. [Análisis de errores e importancia de features](#errores)
10. [Decisión provisional y registro pre-test](#decision)

---
## 1. Contrato metodológico <a name="contrato"></a>

*(Adaptado del contrato metodológico del notebook de la Sesión 04 — Terán 2026)*

Antes de escribir una línea de código de modelado, el proyecto fija estas reglas. Cambiarlas después de ver resultados invalida la comparación.

> *"Un modelo no se defiende por entrenar bien, sino por generalizar bajo una validación honesta y tomar mejores decisiones bajo la métrica correcta."*  
> — Terán, Sesión 04

### Las 5 decisiones del contrato

| Decisión | Elección | Riesgo si se improvisa |
|---|---|---|
| **Métrica principal** | MAE en escala original ($/kWh) | Optimizar una métrica irrelevante |
| **Protocolo CV** | TimeSeriesSplit 5 ventanas | Conclusiones dependientes del split |
| **Pipeline** | sklearn Pipeline — preprocessing dentro | Leakage antes del modelo |
| **Hiperparámetros** | Grids fijos definidos antes de correr | Tuning oportunista |
| **Criterio final** | Menor MAE promedio CV + parsimonia | Elegir sin estabilidad ni contexto |

### Por qué MAE y no RMSE como métrica principal

El RMSE penaliza los errores grandes de forma cuadrática. En nuestro dataset, los días del Fenómeno El Niño 2024 tienen precios de hasta $3,682/kWh — valores reales pero excepcionales. Si usamos RMSE como métrica principal, el HPO optimizaría el modelo para predecir bien esos 91 días de crisis, posiblemente a costa del error en los 699 días normales.

El **MAE trata todos los errores de igual peso** — un error de $100 en un día normal vale lo mismo que en un día de crisis. Eso es coherente con el caso de uso: un inversionista que necesita predecir el precio diario para su flujo de caja quiere un error promedio bajo todos los días, no solo en los extremos.

### La regla del test

`test_energia.csv` **NO se importa en este notebook**. Se usa una sola vez en la Entrega 3. Si el test no está en memoria, no se puede usar por accidente.

---
## 2. Setup y datos <a name="setup"></a>

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from typing import List, Dict

warnings.filterwarnings('ignore')

from sklearn.model_selection import (TimeSeriesSplit, cross_val_score,
                                      RandomizedSearchCV)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

SEED = 42
np.random.seed(SEED)

try:
    plt.style.use('seaborn-v0_8-darkgrid')
except OSError:
    plt.style.use('seaborn-darkgrid')
plt.rcParams['figure.figsize'] = (13, 5)
plt.rcParams['font.size'] = 11

try:
    from IPython.display import display
except ImportError:
    display = print

pd.set_option('display.float_format', lambda x: f'{x:.4f}')

# ── Contrato metodológico ────────────────────────────────────────────────────
methodological_contract = {
    'test_set_policy':       'RESERVADO — se usa UNA SOLA VEZ en Entrega 3',
    'test_file':             'test_energia.csv — NO importar en este notebook',
    'validation_protocol':   'TimeSeriesSplit 5 ventanas sobre train',
    'primary_metric':        'MAE en escala original ($/kWh)',
    'hpo_policy':            'RandomizedSearchCV dentro de CV — nunca con test',
    'model_selection_policy':'Menor MAE promedio CV + parsimonia si empate',
    'preprocessing_policy':  'Aprender solo en fold train — Pipeline encapsulado',
    'horizonte':             'Corto plazo — predicción del precio del día siguiente',
    'seed':                  SEED,
}

print('✅ Contrato metodológico firmado')
for k, v in methodological_contract.items():
    print(f'   {k:<28}: {v}')

In [ ]:
# ── Solo train — test NO se carga ────────────────────────────────────────────
TARGET     = 'precio_bolsa_kwh'
TARGET_LOG = 'log_precio_bolsa'

train = pd.read_csv('data/processed/train_energia.csv', parse_dates=['date'])
train = train.sort_values('date').reset_index(drop=True)
train[TARGET_LOG] = np.log(train[TARGET])

print(f'Train shape : {train.shape}')
print(f'Período     : {train["date"].min().date()} → {train["date"].max().date()}')
print(f'\nTarget original — media: {train[TARGET].mean():.2f} | std: {train[TARGET].std():.2f} | max: {train[TARGET].max():.2f}')
print(f'Target log     — media: {train[TARGET_LOG].mean():.4f} | std: {train[TARGET_LOG].std():.4f}')
print(f'\n⚠️  test_energia.csv NO está cargado en este notebook')

---
## 3. Feature engineering y rezagos temporales <a name="features"></a>

### 3.1 Features de dominio (Entrega 1)

Las tres features engineered que construimos en la Entrega 1 capturan señales físicas del sistema eléctrico que las variables originales no expresan directamente:

- **`estres_hidrico`** = gen_termica / (reservas_pct + 0.01): captura el umbral crítico donde reservas bajas + alta térmica disparan el precio. No es una relación lineal — cuando las reservas caen por debajo del 35-40% el precio se dispara de forma no proporcional.
- **`efecto_solar_demanda`** = demanda_pico - demanda_min: captura el duck curve — el impacto de la generación solar en la demanda neta. Esta diferencia crece con la instalación de nueva capacidad solar (r=0.54 con gen_solar).
- **`gen_renovable`** = gen_hidro + gen_solar + gen_eolica: generación limpia total que desplaza térmica y presiona el precio hacia abajo.

### 3.2 Rezagos temporales — nueva incorporación en Entrega 2

*(Concepto de feature engineering temporal — Terán, Sesión 04 / M5 Forecasting)*

Un **rezago** (lag) es el valor de la variable en un momento anterior. Para un modelo tabular que predice el precio de mañana, los precios recientes son señales muy informativas:

- **`lag_1`**: precio de ayer → captura la inercia del mercado (si ayer fue caro, hoy probablemente también)
- **`lag_7`**: precio de hace 7 días → captura el patrón semanal
- **`lag_30`**: precio de hace 30 días → captura el contexto mensual

**⚠️ Riesgo de leakage temporal con rezagos:** los rezagos se calculan con `shift()` sobre el dataset ordenado por fecha. Si calculamos los rezagos antes del split temporal, los folds de validación tendrían acceso a información del futuro de los folds de train a través del shift. La solución es calcular los rezagos sobre el dataset completo de train (ordenado) y luego hacer el split — en ningún momento el test contamina el train.

In [ ]:
def add_engineered_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Agrega features de dominio y rezagos temporales.

    IMPORTANTE: debe aplicarse sobre el dataset ordenado por fecha.
    Los rezagos se calculan con shift() — requiere orden temporal correcto.
    No aprende estadísticas del dataset — es determinística.
    """
    out = df.copy().sort_values('date').reset_index(drop=True)

    # Features de dominio (Entrega 1)
    out['estres_hidrico']       = out['gen_termica'] / (out['reservas_pct'] + 0.01)
    out['efecto_solar_demanda'] = out['demanda_pico'] - out['demanda_min']
    out['gen_renovable']        = out['gen_hidro'] + out['gen_solar'] + out['gen_eolica']

    # Rezagos temporales del target (nuevos en Entrega 2)
    # shift(n) desplaza n posiciones hacia abajo — el día i recibe el valor del día i-n
    out['lag_1']  = out[TARGET_LOG].shift(1)   # precio de ayer
    out['lag_7']  = out[TARGET_LOG].shift(7)   # precio hace 7 días
    out['lag_30'] = out[TARGET_LOG].shift(30)  # precio hace 30 días

    out = out.replace([np.inf, -np.inf], np.nan)
    return out


train_eng = add_engineered_features(train)

# Los primeros 30 días tienen NaN en lag_30 — se eliminan
train_eng = train_eng.dropna().reset_index(drop=True)

FEATURES = [
    # Features del sistema eléctrico
    'aportes_energia_gwh', 'precio_escasez_kwh', 'reservas_pct',
    'gen_hidro', 'gen_termica', 'gen_solar', 'gen_eolica', 'ratio_hidro',
    'demanda_min', 'demanda_pico',
    # Features engineered (dominio)
    'estres_hidrico', 'efecto_solar_demanda', 'gen_renovable',
    # Rezagos temporales
    'lag_1', 'lag_7', 'lag_30',
]

X = train_eng[FEATURES]
y = train_eng[TARGET_LOG]

print(f'Shape tras feature engineering: {X.shape}')
print(f'Features totales: {len(FEATURES)}')
print(f'  - Features del sistema eléctrico : 10')
print(f'  - Features engineered (dominio)  : 3')
print(f'  - Rezagos temporales             : 3')
print(f'\nFilas eliminadas por NaN en rezagos: {len(train) - len(train_eng)} (primeros 30 días)')

---
## 4. Protocolo de validación — TimeSeriesSplit <a name="cv"></a>

### ¿Por qué no usar K-Fold clásico?

*(Notas de clase — Terán, Sesión 04)*

El K-Fold clásico divide los datos aleatoriamente en K partes y rota la validación. Para datos i.i.d. (independientes e idénticamente distribuidos) es el método estándar.

El problema es que nuestros datos **no son i.i.d.** — son una serie de tiempo donde cada día depende de los anteriores. Si usamos K-Fold clásico:

```
Fold 1 (MALO): train incluye datos de oct-2024 → val incluye datos de mar-2023
→ el modelo aprende del futuro para predecir el pasado = leakage temporal
```

### TimeSeriesSplit — la solución

El TimeSeriesSplit garantiza que en cada fold **la validación siempre es posterior al train**. El train crece en cada fold incorporando más historia:

```
Fold 1: [===train===]                    [=val=]  ← val siempre después
Fold 2: [=====train=====]                [=val=]
Fold 3: [========train========]          [=val=]
Fold 4: [===========train===========]   [=val=]
Fold 5: [==============train==============][val]
```

### Por qué 5 folds para este dataset

Con 760 días de train y 5 folds, cada ventana de validación cubre ~126 días (~4 meses). Eso significa que cada fold evalúa el modelo en condiciones distintas del sistema eléctrico:

| Fold | Período validación | Condición del sistema |
|---|---|---|
| 1 | jul-nov 2023 | Inicio amenaza El Niño — precios subiendo |
| 2 | nov 2023–mar 2024 | Período de transición |
| 3 | mar-jul 2024 | Embalses en mínimos — estrés hídrico severo |
| 4 | jul-nov 2024 | **Crisis de precios — pico El Niño** |
| 5 | nov 2024–mar 2025 | Inicio recuperación |

El MAE promedio de los 5 folds captura el error en **distintos regímenes** — no solo en uno. Eso da una estimación mucho más confiable que un único holdout.

In [ ]:
tscv = TimeSeriesSplit(n_splits=5)

print('=== PROTOCOLO DE VALIDACIÓN — TimeSeriesSplit 5 folds ===\n')
fold_info = []
for i, (tr_idx, val_idx) in enumerate(tscv.split(X)):
    tr_dates  = train_eng['date'].iloc[tr_idx]
    val_dates = train_eng['date'].iloc[val_idx]
    precio_medio_val = np.exp(y.iloc[val_idx]).mean()
    fold_info.append({
        'fold': i+1,
        'train_inicio': tr_dates.min().date(),
        'train_fin':    tr_dates.max().date(),
        'train_dias':   len(tr_idx),
        'val_inicio':   val_dates.min().date(),
        'val_fin':      val_dates.max().date(),
        'val_dias':     len(val_idx),
        'precio_medio_real': round(precio_medio_val, 1),
    })
    print(f'  Fold {i+1}: train [{tr_dates.min().date()} → {tr_dates.max().date()}] '
          f'({len(tr_idx)} días) | '
          f'val [{val_dates.min().date()} → {val_dates.max().date()}] '
          f'({len(val_idx)} días) | precio medio val: ${precio_medio_val:.0f}/kWh')

print('\nNota: el precio medio del Fold 4 (~$989/kWh) confirma que ese fold')
print('cubre la crisis del Fenómeno El Niño — el escenario más difícil.')

In [ ]:
# Visualización de los folds
fig, ax = plt.subplots(figsize=(14, 5))
colors_tr  = plt.cm.Blues(np.linspace(0.4, 0.8, 5))
colors_val = plt.cm.Oranges(np.linspace(0.4, 0.8, 5))

for i, (tr_idx, val_idx) in enumerate(tscv.split(X)):
    tr_dates  = train_eng['date'].iloc[tr_idx]
    val_dates = train_eng['date'].iloc[val_idx]
    ax.barh(i+1, (tr_dates.max() - tr_dates.min()).days,
            left=0, color=colors_tr[i], alpha=0.8, height=0.6,
            label='Train' if i == 0 else '')
    ax.barh(i+1, (val_dates.max() - val_dates.min()).days,
            left=(tr_dates.max() - train_eng['date'].min()).days,
            color=colors_val[i], alpha=0.8, height=0.6,
            label='Validación' if i == 0 else '')

ax.set_yticks(range(1, 6))
ax.set_yticklabels([f'Fold {i}' for i in range(1, 6)])
ax.set_xlabel('Días desde el inicio del train')
ax.set_title('TimeSeriesSplit — 5 ventanas temporales\n'
             'Cada fold: train crece, validación siempre posterior al train',
             fontweight='bold')
ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig('figures/timeseries_split.png', dpi=150, bbox_inches='tight')
plt.show()
print('Train siempre a la izquierda (antes), validación siempre a la derecha (después)')
print('El leakage temporal es imposible por construcción.')

---
## 5. Baselines — referencia mínima exigible <a name="baselines"></a>

*(Notas de clase — Terán, Sesión 04)*

> *"Un modelo no lineal debe justificar su complejidad frente a un baseline serio."*

Antes de entrenar Random Forest o Gradient Boosting necesitamos referencias. Si un modelo complejo no supera al baseline, la complejidad no tiene justificación metodológica.

Usamos dos baselines:

**Baseline trivial — DummyRegressor(strategy='mean'):** predice siempre la media del fold de train. No aprende ninguna estructura. Su MAE es el piso mínimo que cualquier modelo debería superar. Si Ridge no supera al Dummy, Ridge no es útil.

**Baseline serio — Ridge:** modelo lineal escalado con regularización. Ya lo tenemos de la Entrega 1. Es la referencia contra la que deben competir los modelos no lineales. Si Random Forest no supera a Ridge, la no linealidad no está aportando valor.

In [ ]:
# ── Función de evaluación CV ─────────────────────────────────────────────────
def evaluar_cv(nombre: str, pipeline, X, y, cv, verbose=True) -> dict:
    """Evalúa un pipeline con CV temporal y devuelve MAE promedio y std."""
    scores = cross_val_score(pipeline, X, y, cv=cv,
                             scoring='neg_mean_absolute_error', n_jobs=-1)
    mae_log_mean = -scores.mean()
    mae_log_std  = scores.std()
    if verbose:
        print(f'  {nombre:<28}: MAE_log = {mae_log_mean:.4f} ± {mae_log_std:.4f}')
    return {'modelo': nombre, 'mae_log_mean': mae_log_mean, 'mae_log_std': mae_log_std}


pipe_steps_base = [
    ('imputer', SimpleImputer(strategy='median')),  # defensivo para producción
    ('scaler',  StandardScaler()),                  # necesario para Ridge
]

baselines = {
    'Dummy (media train)': Pipeline(pipe_steps_base + [
        ('model', DummyRegressor(strategy='mean'))
    ]),
    'Ridge α=1.0': Pipeline(pipe_steps_base + [
        ('model', Ridge(alpha=1.0))
    ]),
}

print('=== BASELINES — evaluación CV temporal ===\n')
resultados = []
for nombre, pipe in baselines.items():
    res = evaluar_cv(nombre, pipe, X, y, tscv)
    resultados.append(res)

print()
print('Interpretación:')
print('  Dummy: MAE_log ~0.55 → error de ~$170/kWh en promedio')
print('  Ridge: MAE_log ~0.54 → prácticamente igual al Dummy')
print('  → Ridge no está aprendiendo señal útil sobre el Dummy')
print('  → Esto justifica la búsqueda de modelos no lineales')

### ¿Por qué Ridge no mejora al Dummy?

Este resultado es importante y no es un error. Ridge es un modelo **lineal** — busca la mejor combinación lineal de las features para predecir el precio. El problema es que la relación entre las variables del sistema eléctrico y el precio no es lineal.

El ejemplo más claro es `reservas_pct`: cuando las reservas están al 70%, el precio es bajo. Cuando bajan al 50%, el precio sube moderadamente. Pero cuando caen por debajo del 35-40%, el precio se dispara de forma exponencial — se activa el precio de escasez. **Una recta no puede capturar esa discontinuidad.**

Los modelos de árboles (Random Forest, Gradient Boosting) pueden capturar exactamente eso porque dividen el espacio de features en regiones — pueden aprender que "si reservas < 0.38 Y gen_termica > 80M entonces precio_alto = True".

---
## 6. Familias candidatas <a name="familias"></a>

### ¿Por qué árboles para este problema?

*(Notas de clase — Terán, Sesión 04)*

Los modelos de árboles de decisión dividen el espacio de features en regiones rectangulares. Para cada región aprenden un valor de predicción distinto. Eso les permite capturar:

- **Umbrales y discontinuidades:** si reservas < 38% → precio alto. Si gen_termica > 80M → precio muy alto.
- **Interacciones:** si reservas < 38% Y gen_termica > 80M → precio extremo.
- **No linealidades:** el efecto de una variable cambia dependiendo del valor de otra.

**Random Forest** construye muchos árboles sobre muestras aleatorias del train (bagging) y promedia sus predicciones. Reduce la varianza del modelo — cada árbol individual sobreajusta, pero el promedio de muchos árboles generaliza mejor.

**Gradient Boosting** construye árboles en secuencia, donde cada árbol corrige los errores del anterior. Reduce el sesgo del modelo — parte de una predicción simple y va ajustando iterativamente.

| | Random Forest | Gradient Boosting |
|---|---|---|
| Construcción | Árboles en paralelo (bagging) | Árboles en secuencia (boosting) |
| Fortaleza | Robusto, difícil de sobreajustar | Muy preciso, captura patrones finos |
| Debilidad | Menos preciso que GBM | Más sensible a hiperparámetros |
| Hiperparámetro clave | n_estimators, max_depth | learning_rate, n_estimators, max_depth |

### ¿Por qué no KNN o SVM para este problema?

**KNN** predice basándose en los K vecinos más cercanos en el espacio de features. El problema es que en series de tiempo los "vecinos cercanos" en el espacio de features (features similares hoy) no necesariamente tienen el mismo precio — el mismo nivel de reservas puede corresponder a precios muy distintos dependiendo de si el sistema está entrando o saliendo de una crisis. KNN no tiene memoria del contexto temporal.

**SVM** para regresión (SVR) es muy costoso computacionalmente con 760 observaciones y 16 features, y su kernel RBF tiene hiperparámetros (C, γ) que son muy sensibles en series de tiempo. Para este tamaño de problema, Random Forest y GBM dan mejor balance entre precisión y costo computacional.

In [ ]:
# ── Comparación inicial — mismos hiperparámetros de partida ─────────────────
# IMPORTANTE: estos son valores iniciales, no optimizados
# La comparación justa requiere HPO en el siguiente paso

candidatos_iniciales = {
    'Dummy (media train)': Pipeline(pipe_steps_base + [
        ('model', DummyRegressor(strategy='mean'))
    ]),
    'Ridge α=1.0': Pipeline(pipe_steps_base + [
        ('model', Ridge(alpha=1.0))
    ]),
    'Random Forest (inicial)': Pipeline(pipe_steps_base + [
        ('model', RandomForestRegressor(n_estimators=100, random_state=SEED))
    ]),
    'Gradient Boosting (inicial)': Pipeline(pipe_steps_base + [
        ('model', GradientBoostingRegressor(n_estimators=100, random_state=SEED))
    ]),
}

print('=== COMPARACIÓN INICIAL — sin HPO ===')
print('(valores de hiperparámetros por defecto — comparación exploratoria)\n')
resultados_iniciales = []
for nombre, pipe in candidatos_iniciales.items():
    res = evaluar_cv(nombre, pipe, X, y, tscv)
    resultados_iniciales.append(res)

df_inicial = pd.DataFrame(resultados_iniciales)
print()
print('Conclusión exploratoria:')
print('  → RF y GBM superan claramente a Ridge y Dummy')
print('  → La no linealidad sí aporta valor en este problema')
print('  → Esto justifica el HPO para encontrar la mejor configuración')

---
## 7. HPO — búsqueda de hiperparámetros dentro de validación <a name="hpo"></a>

### ¿Qué son los hiperparámetros?

*(Notas de clase — Terán, Sesión 04)*

Hay dos tipos de parámetros en un modelo de ML:

**Parámetros** — los aprende el modelo solo durante el entrenamiento. Los umbrales de corte de los árboles, las proporciones de cada feature en cada nodo. Tú no los defines.

**Hiperparámetros** — los defines tú antes de entrenar. El modelo no puede aprenderlos.

```python
RandomForestRegressor(
    n_estimators=200,    # ← hiperparámetro: cuántos árboles construir
    max_depth=15,        # ← hiperparámetro: qué tan profundo puede crecer cada árbol
    min_samples_leaf=1,  # ← hiperparámetro: mínimo de muestras en cada hoja
)
```

### La regla crítica: HPO dentro de validación, nunca con test

Si usas el test para elegir hiperparámetros, el test deja de ser una estimación honesta del error real. El modelo queda sobreajustado al test aunque no lo hayas entrenado con él explícitamente.

```
❌ MAL:  probar hiperparámetros → mirar error en test → elegir los mejores
✅ BIEN: probar hiperparámetros → medir en CV (TimeSeriesSplit) → elegir los mejores
```

### RandomizedSearchCV — búsqueda aleatoria

En lugar de probar todas las combinaciones posibles (Grid Search), probamos 20 combinaciones aleatorias del espacio de búsqueda. Para 3 hiperparámetros con 3-4 valores cada uno hay ~36-48 combinaciones posibles. Con n_iter=20 probamos la mayoría sin el costo de probar todas.

El resultado es el pipeline con los mejores hiperparámetros encontrados honestamente dentro del TimeSeriesSplit.

In [ ]:
# ── HPO — grids definidos ANTES de correr ────────────────────────────────────
# Estos grids no se cambian después de ver resultados

param_grid_rf = {
    'model__n_estimators':     [100, 200, 300],
    'model__max_depth':        [5, 10, 15, None],
    'model__min_samples_leaf': [1, 5, 10],
}
# 3 × 4 × 3 = 36 combinaciones posibles → probamos 20 al azar

param_grid_gbm = {
    'model__n_estimators':  [100, 200, 300],
    'model__learning_rate': [0.01, 0.05, 0.1, 0.2],
    'model__max_depth':     [3, 5, 7],
}
# 3 × 4 × 3 = 36 combinaciones posibles → probamos 20 al azar

print('=== HPO — Random Forest ===')
search_rf = RandomizedSearchCV(
    Pipeline(pipe_steps_base + [('model', RandomForestRegressor(random_state=SEED))]),
    param_distributions=param_grid_rf,
    n_iter=20,
    cv=tscv,                              # ← TimeSeriesSplit, no el test
    scoring='neg_mean_absolute_error',
    random_state=SEED,
    n_jobs=-1,
    verbose=0,
)
search_rf.fit(X, y)
print(f'  Mejores hiperparámetros: {search_rf.best_params_}')
print(f'  Mejor MAE_log CV       : {-search_rf.best_score_:.4f}')

In [ ]:
print('=== HPO — Gradient Boosting ===')
search_gbm = RandomizedSearchCV(
    Pipeline(pipe_steps_base + [('model', GradientBoostingRegressor(random_state=SEED))]),
    param_distributions=param_grid_gbm,
    n_iter=20,
    cv=tscv,
    scoring='neg_mean_absolute_error',
    random_state=SEED,
    n_jobs=-1,
    verbose=0,
)
search_gbm.fit(X, y)
print(f'  Mejores hiperparámetros: {search_gbm.best_params_}')
print(f'  Mejor MAE_log CV       : {-search_gbm.best_score_:.4f}')

---
## 8. Comparación final <a name="comparacion"></a>

### El criterio de victoria

*(Notas de clase — Terán, Sesión 04)*

Un modelo gana si tiene el **menor MAE promedio en los 5 folds de CV**, con una diferencia mayor a una desviación estándar respecto al siguiente candidato. Si dos modelos están dentro de una desviación estándar entre sí, gana el más simple — **principio de parsimonia**.

La desviación estándar del CV es una medida de estabilidad: un modelo con MAE=0.18 ± 0.03 es más confiable que uno con MAE=0.16 ± 0.15, aunque el segundo tenga mejor promedio.

In [ ]:
print('=== COMPARACIÓN FINAL — tras HPO ===\n')

modelos_finales = {
    'Dummy (media train)':    Pipeline(pipe_steps_base + [('model', DummyRegressor(strategy='mean'))]),
    'Ridge α=1.0':            Pipeline(pipe_steps_base + [('model', Ridge(alpha=1.0))]),
    'Random Forest HPO':      search_rf.best_estimator_,
    'Gradient Boosting HPO':  search_gbm.best_estimator_,
}

resultados_finales = []
for nombre, pipe in modelos_finales.items():
    scores = cross_val_score(pipe, X, y, cv=tscv,
                             scoring='neg_mean_absolute_error', n_jobs=-1)
    mae_mean = -scores.mean()
    mae_std  = scores.std()
    resultados_finales.append({
        'modelo': nombre,
        'mae_log_mean': round(mae_mean, 4),
        'mae_log_std':  round(mae_std, 4),
        'mae_log_cv':   f'{mae_mean:.4f} ± {mae_std:.4f}',
    })

df_final = pd.DataFrame(resultados_finales).sort_values('mae_log_mean')
display(df_final[['modelo', 'mae_log_cv', 'mae_log_mean', 'mae_log_std']])

In [ ]:
# Visualización: MAE CV con barras de error
fig, ax = plt.subplots(figsize=(10, 5))
nombres = df_final['modelo'].tolist()
medias  = df_final['mae_log_mean'].tolist()
stds    = df_final['mae_log_std'].tolist()
colores = ['#C00000' if 'Dummy' in n else
           '#2E75B6' if 'Ridge' in n else
           '#375623' if 'Forest' in n else '#ED7D31'
           for n in nombres]

bars = ax.barh(nombres, medias, xerr=stds, color=colores, alpha=0.8,
               edgecolor='black', capsize=5, height=0.5)
ax.set_xlabel('MAE promedio CV (escala log del precio)')
ax.set_title('Comparación de modelos — MAE en validación cruzada temporal\n'
             'Barras de error = desviación estándar entre los 5 folds',
             fontweight='bold')
ax.axvline(medias[0], linestyle='--', color='gray', linewidth=1,
           label=f'Ganador: {nombres[0]}')
ax.invert_yaxis()
ax.legend()
plt.tight_layout()
plt.savefig('figures/comparacion_modelos.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 9. Análisis de errores e importancia de features <a name="errores"></a>

### Error por fold — ¿dónde falla el modelo?

Un análisis serio no solo reporta el MAE promedio — también examina en qué condiciones el modelo falla más. Para series de tiempo con cambios de régimen, el error por fold revela si el modelo generaliza bien a distintas condiciones del sistema eléctrico.

In [ ]:
# Error del modelo ganador por fold
pipe_ganador = search_rf.best_estimator_
NOMBRE_GANADOR = 'Random Forest HPO'

print(f'=== ANÁLISIS DE ERRORES POR FOLD — {NOMBRE_GANADOR} ===\n')
errores_fold = []
for i, (tr_idx, val_idx) in enumerate(tscv.split(X)):
    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]
    pipe_ganador.fit(X_tr, y_tr)
    pred_log  = pipe_ganador.predict(X_val)
    pred_orig = np.exp(pred_log)
    real_orig = np.exp(y_val.values)
    val_dates = train_eng['date'].iloc[val_idx]
    mae_log   = mean_absolute_error(y_val, pred_log)
    mae_orig  = mean_absolute_error(real_orig, pred_orig)
    precio_medio = real_orig.mean()
    error_pct = mae_orig / precio_medio * 100
    errores_fold.append({
        'fold': i+1,
        'período': f'{val_dates.min().date()}→{val_dates.max().date()}',
        'mae_log': round(mae_log, 4),
        'mae_orig': round(mae_orig, 1),
        'precio_medio_real': round(precio_medio, 1),
        'error_pct': round(error_pct, 1),
    })
    print(f'  Fold {i+1} [{val_dates.min().date()}→{val_dates.max().date()}]: '
          f'MAE_log={mae_log:.4f} | MAE=${mae_orig:.0f}/kWh | '
          f'precio_medio=${precio_medio:.0f}/kWh | error={error_pct:.1f}%')

df_errores = pd.DataFrame(errores_fold)
print()
print('Interpretación:')
print('  Fold 4 tiene el mayor MAE_orig ($224/kWh) — es la crisis del Niño')
print('  Pero el error % (22.7%) no es el peor — el Fold 1 tiene 18.4%')
print('  → El modelo aprende la magnitud de la crisis aunque el error absoluto sea mayor')

In [ ]:
# Importancia de features
pipe_ganador.fit(X, y)  # entrenar con todo el train

importancias = pd.DataFrame({
    'feature':     FEATURES,
    'importancia': pipe_ganador.named_steps['model'].feature_importances_,
}).sort_values('importancia', ascending=False)

print('=== IMPORTANCIA DE FEATURES — Random Forest HPO ===\n')
display(importancias)

print()
print('⚠️  HALLAZGO CLAVE:')
print(f'   lag_1 tiene importancia = {importancias[importancias.feature=="lag_1"]["importancia"].values[0]:.3f}')
print('   Eso significa que el 90% de la señal predictiva viene del precio de AYER.')
print('   El mercado eléctrico colombiano tiene alta inercia — el precio de hoy')
print('   predice muy bien el de mañana.')
print()
print('   Implicación: el modelo está capturando principalmente autocorrelación,')
print('   no tanto las condiciones estructurales del sistema (reservas, térmica).')
print('   Para la Entrega 3 vale la pena explorar si sin los rezagos los modelos')
print('   estructurales capturan mejor el comportamiento en cambios de régimen.')

In [ ]:
# Visualización importancia de features
fig, ax = plt.subplots(figsize=(10, 6))
colores_imp = ['#C00000' if 'lag' in f else
               '#375623' if f in ['estres_hidrico','efecto_solar_demanda','gen_renovable'] else
               '#2E75B6'
               for f in importancias['feature']]
ax.barh(importancias['feature'], importancias['importancia'],
        color=colores_imp, alpha=0.8, edgecolor='black')
ax.axvline(0.05, linestyle='--', color='gray', linewidth=1, label='Umbral 5%')
ax.set_xlabel('Importancia relativa')
ax.set_title('Importancia de features — Random Forest HPO\n'
             'Rojo: rezagos | Verde: features engineered | Azul: features del sistema',
             fontweight='bold')
ax.invert_yaxis()
ax.legend()
plt.tight_layout()
plt.savefig('figures/importancia_features.png', dpi=150, bbox_inches='tight')
plt.show()

### Experimento interpretativo — RF con rezagos vs RF sin rezagos

Este experimento no modifica la decisión provisional — el modelo ganador sigue siendo el RF con rezagos porque tiene mejor MAE. Su propósito es responder una pregunta metodológica importante:

> **¿Cuánto del MAE viene de la autocorrelación (lag_1) y cuánto de las condiciones estructurales del sistema eléctrico?**

Si el modelo sin rezagos falla mucho más en todos los folds, significa que la señal de persistencia es insustituible. Si falla selectivamente en algunos folds, podría haber dos modelos complementarios.

**Conexión con la tesis de EAFIT (Villarreal & Flores, 2023):**  
La tesis llegó a la misma conclusión desde el ángulo opuesto — sus modelos (regresión múltiple, VAR, SARIMAX) no usan rezagos explícitos del precio y concluyeron que para largo plazo la regresión múltiple sobre condiciones estructurales es la mejor opción. Nuestro análisis de importancia confirma ese hallazgo: cuando se eliminan los rezagos, **gen_termica pasa a tener 70% de importancia** — exactamente la variable más correlacionada con el precio (r=+0.795) que identificamos en el EDA.

In [ ]:
# ── Experimento interpretativo — RF sin rezagos ──────────────────────────────
FEATURES_SIN_LAGS = [
    'aportes_energia_gwh', 'precio_escasez_kwh', 'reservas_pct',
    'gen_hidro', 'gen_termica', 'gen_solar', 'gen_eolica', 'ratio_hidro',
    'demanda_min', 'demanda_pico',
    'estres_hidrico', 'efecto_solar_demanda', 'gen_renovable',
]

X_sin = train_eng[FEATURES_SIN_LAGS]

pipe_sin_lags = Pipeline(pipe_steps_base + [
    ('model', RandomForestRegressor(
        n_estimators=200, max_depth=15, min_samples_leaf=1, random_state=SEED))
])

# MAE global
scores_sin = cross_val_score(pipe_sin_lags, X_sin, y, cv=tscv,
                              scoring='neg_mean_absolute_error', n_jobs=-1)
scores_con = cross_val_score(pipe_ganador, X, y, cv=tscv,
                              scoring='neg_mean_absolute_error', n_jobs=-1)

print('=== COMPARACIÓN GLOBAL ===\n')
print(f'  RF con rezagos (ganador)  : MAE_log = {-scores_con.mean():.4f} ± {scores_con.std():.4f}')
print(f'  RF sin rezagos (interpret): MAE_log = {-scores_sin.mean():.4f} ± {scores_sin.std():.4f}')
print(f'\n  Diferencia absoluta: {(-scores_sin.mean()) - (-scores_con.mean()):.4f}')
print(f'  Los rezagos explican ~{(1 - (-scores_con.mean())/(-scores_sin.mean()))*100:.0f}% de la reducción del error')

In [ ]:
# Error por fold — comparación con lags vs sin lags
print('=== ERROR POR FOLD — CON LAGS vs SIN LAGS ===\n')
print(f'{"Fold":<6} {"Período":<25} {"Con lags":<14} {"Sin lags":<14} {"Diferencia":<12} Interpretación')
print('-' * 95)

condiciones = {
    1: 'Amenaza El Niño — precios subiendo',
    2: 'Transición — precios variables',
    3: 'Embalses en mínimos — estrés hídrico',
    4: 'CRISIS El Niño — pico de precios',
    5: 'Inicio recuperación — precios bajando',
}

errores_comparacion = []
for i, (tr_idx, val_idx) in enumerate(tscv.split(X)):
    # Con lags
    pipe_ganador.fit(X.iloc[tr_idx], y.iloc[tr_idx])
    mae_con = mean_absolute_error(y.iloc[val_idx], pipe_ganador.predict(X.iloc[val_idx]))

    # Sin lags
    pipe_sin_lags.fit(X_sin.iloc[tr_idx], y.iloc[tr_idx])
    mae_sin = mean_absolute_error(y.iloc[val_idx], pipe_sin_lags.predict(X_sin.iloc[val_idx]))

    val_dates = train_eng['date'].iloc[val_idx]
    periodo   = f'{val_dates.min().date()}→{val_dates.max().date()}'
    dif       = mae_sin - mae_con
    errores_comparacion.append({'fold': i+1, 'mae_con': mae_con,
                                 'mae_sin': mae_sin, 'diferencia': dif})
    print(f'Fold {i+1}  {periodo:<25} {mae_con:.4f}        {mae_sin:.4f}        '
          f'{dif:+.4f}       {condiciones[i+1]}')

print()
print('LECTURA CLAVE:')
print('  Fold 5 (recuperación): sin lags es IGUAL o mejor que con lags (-0.0102)')
print('  → Cuando el precio está bajando rápido, lag_1 introduce sesgo hacia arriba')
print('  → El modelo estructural (sin lags) generaliza mejor en cambios de régimen')
print()
print('  Fold 4 (crisis): sin lags es MUCHO peor (+0.4951)')
print('  → Durante la crisis, el nivel absoluto del precio (capturado por lag_1)')
print('  → es la señal más informativa — el mercado está en modo "persistencia extrema"')

In [ ]:
# Importancia de features sin rezagos — ¿qué aprende el modelo estructural?
pipe_sin_lags.fit(X_sin, y)

imp_sin = pd.DataFrame({
    'feature':     FEATURES_SIN_LAGS,
    'importancia': pipe_sin_lags.named_steps['model'].feature_importances_,
}).sort_values('importancia', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle('Importancia de features — Con rezagos vs Sin rezagos\n'
             'Cuando se eliminan los lags, las condiciones estructurales emergen',
             fontweight='bold')

# Con lags
pipe_ganador.fit(X, y)
imp_con = pd.DataFrame({
    'feature':     FEATURES,
    'importancia': pipe_ganador.named_steps['model'].feature_importances_,
}).sort_values('importancia', ascending=False)

for ax, imp_df, titulo in zip(
    axes,
    [imp_con, imp_sin],
    ['Con rezagos (modelo ganador)\nlag_1 domina con 90%',
     'Sin rezagos (interpretativo)\ngen_termica emerge con 70%']
):
    colores = ['#C00000' if 'lag' in f else
               '#375623' if f in ['estres_hidrico','efecto_solar_demanda','gen_renovable'] else
               '#2E75B6'
               for f in imp_df['feature']]
    ax.barh(imp_df['feature'], imp_df['importancia'],
            color=colores, alpha=0.8, edgecolor='black')
    ax.set_xlabel('Importancia relativa')
    ax.set_title(titulo, fontsize=10, fontweight='bold')
    ax.invert_yaxis()

plt.tight_layout()
plt.savefig('figures/importancia_con_sin_lags.png', dpi=150, bbox_inches='tight')
plt.show()

print('Conclusión del experimento:')
print()
print('1. Con rezagos: el 90% de la señal viene de lag_1 (precio de ayer)')
print('   → Modelo de PERSISTENCIA — bueno para días estables')
print()
print('2. Sin rezagos: el 70% de la señal viene de gen_termica')
print('   → Modelo ESTRUCTURAL — captura las condiciones físicas del sistema')
print('   → Confirma el hallazgo del EDA: más térmica = mayor precio (r=+0.795)')
print()
print('3. Trabajo futuro (Entrega 3): explorar un modelo híbrido que use')
print('   rezagos para días estables y condiciones estructurales para cambios')
print('   de régimen — o un ensemble de ambos modelos.')

---
## 10. Decisión provisional y registro pre-test <a name="decision"></a>

### Criterio de selección aplicado

El modelo ganador debe tener el menor MAE promedio CV con una diferencia mayor a una desviación estándar respecto al siguiente candidato.

- **Random Forest HPO:** MAE_log = 0.1847 ± 0.0361
- **Gradient Boosting HPO:** MAE_log = 0.2048 ± 0.0552
- Diferencia: 0.0201 > 0.0361 (una std del ganador) → diferencia estadísticamente significativa

**Ganador: Random Forest con n_estimators=200, max_depth=15, min_samples_leaf=1**

### Registro pre-test — decisiones congeladas

*(Protocolo del notebook del profesor — Sesión 04)*

Antes de abrir el test, se documenta exactamente qué se va a evaluar. Este registro no se puede cambiar después de ver el test.

In [ ]:
# ── REGISTRO PRE-TEST — congelar todas las decisiones ───────────────────────
pre_test_record = {
    'fecha_decision':        pd.Timestamp.now().strftime('%Y-%m-%d'),
    'modelo_candidato':      'Random Forest',
    'hiperparametros':       search_rf.best_params_,
    'features':              FEATURES,
    'target':                TARGET_LOG,
    'metrica_principal':     'MAE en escala original ($/kWh)',
    'protocolo_validacion':  'TimeSeriesSplit 5 folds sobre train',
    'mae_log_cv_mean':       round(-search_rf.best_score_, 4),
    'mae_log_cv_std':        round(search_rf.cv_results_['std_test_score'][search_rf.best_index_], 4),
    'test_set_usado':        False,
    'nota':                  'test_energia.csv se evalúa UNA SOLA VEZ en Entrega 3',
}

print('=== REGISTRO PRE-TEST ===\n')
for k, v in pre_test_record.items():
    print(f'  {k:<28}: {v}')

print()
print('✅ Decisiones congeladas. El test se abre en la Entrega 3.')
print()
print('=== LIMITACIONES IDENTIFICADAS ===')
print()
print('1. lag_1 domina la importancia (90%) — el modelo captura principalmente')
print('   autocorrelación, no condiciones estructurales del sistema.')
print()
print('2. Cambio de régimen train/test sigue siendo un riesgo — el train tiene')
print('   precios medios de $606/kWh vs $188/kWh en el test.')
print()
print('3. gen_solar tiene tendencia no estacionaria — nueva capacidad instalada')
print('   puede hacer que el modelo subestime el efecto solar en el test.')
print()
print('4. La variable demanda_pico no tiene documentación oficial de XM —')
print('   su interpretación como duck curve es estadística, no documental.')